# Matemática IV

# Trabajo Práctico Final: Ciencia de Datos

## Análisis de Datos del Transporte Aéreo en Argentina

**Integrantes:**  
- Baselli María  
- Fernández Emiliano  
- Letelle Mauro

# Índice

1. Introducción
2. Fuente de datos
3. Objetivos y preguntas de análisis
4. Metodología y decisiones de limpieza
5. Carga de datos
6. Análisis por preguntas
7. Conclusiones finales
8. Limitaciones
9. Bibliografía

# Introducción

El transporte aéreo cumple un rol central en la conectividad de Argentina: reduce distancias, integra regiones, sostiene actividad turística y económica, y permite observar patrones de movilidad que no siempre son visibles en otros medios de transporte. Analizar sus datos permite responder preguntas concretas sobre demanda, ocupación, rutas relevantes y puntualidad del servicio.

Este trabajo estudia el comportamiento reciente del mercado aerocomercial argentino a partir de tres fuentes locales: conectividad aérea, movimientos aeroportuarios de ANAC e información de demoras publicada por Failbondi. El objetivo no es solo describir los datos, sino transformar esos resultados en conclusiones útiles para tomar decisiones operativas: dónde hay mayor demanda, cuándo se concentran los pasajeros, qué rutas sostienen más tráfico, qué servicios tienen alta ocupación y dónde aparecen más demoras.

Todo el análisis se ejecuta localmente desde este repositorio. Los archivos crudos se encuentran en `datasets/` y el material de limpieza previo queda como referencia en `Datasets ANALIZADOS/`.

# Fuente de datos

El notebook utiliza tres conjuntos de datos locales:

1. **Conectividad aérea**  
   Archivos: `conectividad_aerea_cabotaje.csv` y `conectividad_aerea_internacional.csv`.  
   Contienen vuelos, pasajeros, asientos, aerolíneas, rutas, origen, destino, clase de vuelo y fecha. En los archivos disponibles cubren el período **2019-2026**.

2. **Aterrizajes y despegues ANAC**  
   Archivos: `202512-informe-ministerio-actualizado-dic-final.csv` y `202604-informe-ministerio.csv`.  
   Contienen movimientos aeroportuarios, `Pasajeros`, `PAX`, fecha, hora, aeropuerto, aerolínea, tipo de movimiento y clasificación del vuelo. En ANAC, `Pasajeros` y `PAX` no son equivalentes: `Pasajeros` mide personas por movimiento aeroportuario y `PAX` se usa como pasajeros equivalentes para demanda agregada. Con los CSV locales actuales cubren **2025-2026**. Para análisis por hora, día y mes se usa **2025 completo**, porque 2026 está incompleto.

3. **Failbondi**  
   Archivo: `failbondi-2026-05-20-dump.json`.  
   Contiene vuelos, aerolínea, aeropuerto, movimiento, estado, matrícula, aeronave, fecha programada y diferencia temporal respecto de la operación prevista. Como el dump fue generado el **20/05/2026**, se recortan vuelos posteriores a esa fecha para no analizar vuelos futuros como si fueran observados.

Fuentes mencionadas:

- Tableros Yvera: https://tableros.yvera.tur.ar/conectividad/
- Datos abiertos ANAC: https://www.datos.gob.ar/
- Failbondi: https://failbondi.fail/acerca

# Objetivos y preguntas de análisis

## Objetivo general

Analizar datos del transporte aéreo argentino para identificar patrones de demanda, ocupación y demoras que ayuden a comprender el funcionamiento del sistema y a proponer decisiones operativas basadas en datos.

## Preguntas de análisis

1. ¿Cuál es el volumen y alcance temporal real de cada fuente?
2. ¿En qué horarios se concentra la mayor y menor demanda de pasajeros equivalentes?
3. ¿Qué días de la semana tienen más movimiento?
4. ¿Qué meses presentan mayor y menor cantidad de pasajeros equivalentes?
5. ¿Qué rutas concentran más pasajeros en cabotaje, internacional y total?
6. ¿Cómo se comporta la ocupación y qué criterio se usa para definir alta ocupación?
7. ¿Qué valores atípicos aparecen en la serie histórica y cómo se interpretan?
8. ¿Cuál es el comportamiento general de las demoras?
9. ¿Las demoras cambian según tipo de movimiento y aerolínea?
10. ¿Las demoras varían según hora del día y día de semana?
11. ¿Qué probabilidades pueden estimarse con modelos binomial y Poisson?

# Metodología y decisiones de limpieza

- El análisis usa rutas locales del repositorio y no depende de servicios externos.
- Para ANAC se usa 2025 completo en gráficos por hora, día y mes. Los datos de 2026 se mantienen para inspección, pero no para comparar años completos.
- En ANAC `Pasajeros` y `PAX` no son lo mismo. `Pasajeros` cuenta personas asociadas a cada movimiento aeroportuario (`Aterrizaje` o `Despegue`). `PAX` se usa como pasajeros equivalentes para demanda agregada: coincide con `Pasajeros` en vuelos internacionales, pero en vuelos domésticos suele ser aproximadamente la mitad para evitar duplicar cabotaje al sumar demanda.
- Failbondi se recorta hasta el 20/05/2026, fecha del dump local, para evitar vuelos futuros programados.
- Se considera demora relevante cuando `delta_minutos > 15`.
- Para limpiar demoras extremas se conservan vuelos entre `-180` y `360` minutos. El recorte evita que valores aislados dominen promedios y gráficos.
- La ocupación se calcula como `Pasajeros / Asientos`, porque mide qué proporción de asientos se ocupó en el movimiento registrado.
- Los nulos de ocupación no se pisan: se conserva `ocupacion_original`, se marca `ocupacion_fue_imputada` y se crea `ocupacion` imputada con la media de su clasificación.
- La alta ocupación no usa un umbral arbitrario. Se define por clasificación: un registro tiene alta ocupación si su ocupación es mayor o igual a la mediana de su grupo (`Cabotaje` o `Internacional`).

# Carga de librerías y rutas locales

In [ ]:
from pathlib import Path
from math import comb
import json
import os

Path('.matplotlib').mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import poisson

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
plt.style.use('seaborn-v0_8-whitegrid')

ARCHIVOS_REQUERIDOS = [
    'conectividad_aerea_cabotaje.csv',
    'conectividad_aerea_internacional.csv',
    '202512-informe-ministerio-actualizado-dic-final.csv',
    '202604-informe-ministerio.csv',
    'failbondi-2026-05-20-dump.json',
]

def resolver_base_dir():
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'datasets').is_dir():
            return carpeta
    raise FileNotFoundError(
        'No se encontró la carpeta datasets/. Abrí VS Code en la raíz del proyecto MateIV_Transporte_Publico.'
    )

BASE_DIR = resolver_base_dir()
DATA_DIR = BASE_DIR / 'datasets'
ANALIZADOS_DIR = BASE_DIR / 'Datasets ANALIZADOS'

faltantes = [archivo for archivo in ARCHIVOS_REQUERIDOS if not (DATA_DIR / archivo).is_file()]
if faltantes:
    mensaje = 'Faltan archivos en datasets/:' + chr(10) + '- ' + (chr(10) + '- ').join(faltantes)
    raise FileNotFoundError(mensaje)

print('Carpeta base:', BASE_DIR)
print('Carpeta datasets:', DATA_DIR)
print('Carpeta datasets analizados:', ANALIZADOS_DIR)
print('Archivos encontrados:')
for archivo in sorted(p.name for p in DATA_DIR.iterdir() if p.is_file()):
    print('-', archivo)

# Carga y preparación de datos

In [ ]:
MESES = {
    'Enero': 1,
    'Febrero': 2,
    'Marzo': 3,
    'Abril': 4,
    'Mayo': 5,
    'Junio': 6,
    'Julio': 7,
    'Agosto': 8,
    'Septiembre': 9,
    'Octubre': 10,
    'Noviembre': 11,
    'Diciembre': 12,
}

DIAS = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miércoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo',
}

ORDEN_DIAS = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
FECHA_CORTE_FAILBONDI = pd.Timestamp('2026-05-20 23:59:59', tz='UTC')


def leer_conectividad(nombre_archivo, clasificacion):
    df = pd.read_csv(DATA_DIR / nombre_archivo)
    df.columns = df.columns.str.strip()
    df['Clasificacion'] = clasificacion
    df['mes_num'] = df['Mes'].map(MESES)
    df['fecha'] = pd.to_datetime(
        dict(year=df['Año'], month=df['mes_num'], day=df['Dia']),
        errors='coerce'
    )
    df['anio'] = df['fecha'].dt.year
    df['ocupacion_original'] = np.where(df['Asientos'] > 0, df['Pasajeros'] / df['Asientos'], np.nan)
    return df


def leer_anac(nombre_archivo):
    df = pd.read_csv(DATA_DIR / nombre_archivo, sep=';', decimal=',', on_bad_lines='warn')
    df.columns = df.columns.str.strip()

    for columna in ['Pasajeros', 'PAX']:
        df[columna] = pd.to_numeric(df[columna], errors='coerce')

    df['pasajeros_movimiento'] = df['Pasajeros']
    df['pasajeros_equivalentes'] = df['PAX']

    df['Fecha UTC'] = pd.to_datetime(df['Fecha UTC'], dayfirst=True, errors='coerce')
    df['anio'] = df['Fecha UTC'].dt.year
    df['mes'] = df['Fecha UTC'].dt.month
    df['dia_semana'] = df['Fecha UTC'].dt.day_name().map(DIAS)
    df['hora'] = pd.to_datetime(df['Hora UTC'], format='%H:%M', errors='coerce').dt.hour
    return df


def cargar_failbondi(path, chunk_size=1024 * 1024):
    registros = []
    decoder = json.JSONDecoder()
    buffer = ''
    dentro_array = False
    fin = False

    with open(path, encoding='utf-8', errors='replace') as archivo:
        while not fin:
            chunk = archivo.read(chunk_size)
            if chunk == '':
                fin = True
            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not dentro_array:
                    if not buffer:
                        break
                    if buffer[0] != '[':
                        raise ValueError('El JSON de Failbondi no empieza con un array.')
                    buffer = buffer[1:]
                    dentro_array = True
                    buffer = buffer.lstrip()

                if buffer.startswith(']'):
                    fin = True
                    break

                if buffer.startswith(','):
                    buffer = buffer[1:].lstrip()

                if not buffer:
                    break

                try:
                    item, idx = decoder.raw_decode(buffer)
                except json.JSONDecodeError:
                    break

                raw = item.get('json') or {}
                registros.append({
                    'flight_id': item.get('aerolineas_flight_id'),
                    'fecha_programada': item.get('stda_parsed'),
                    'aeropuerto': raw.get('arpt'),
                    'movimiento': raw.get('mov'),
                    'nro_vuelo': raw.get('nro'),
                    'aerolinea': raw.get('aerolinea'),
                    'destino_origen': raw.get('destorig'),
                    'estado': raw.get('estes'),
                    'matricula': item.get('matricula'),
                    'aeronave': item.get('aeronave'),
                    'edad_avion': item.get('edad_del_avion'),
                    'delta_segundos': item.get('delta'),
                })
                buffer = buffer[idx:]

    df = pd.DataFrame(registros)
    df['fecha_programada'] = pd.to_datetime(df['fecha_programada'], errors='coerce')
    df['delta_segundos'] = pd.to_numeric(df['delta_segundos'], errors='coerce')
    df['delta_minutos'] = df['delta_segundos'] / 60
    df['demorado_15m'] = df['delta_minutos'] > 15
    df['adelantado_5m'] = df['delta_minutos'] < -5
    df['fecha'] = df['fecha_programada'].dt.date
    df['hora'] = df['fecha_programada'].dt.hour
    df['dia_semana'] = df['fecha_programada'].dt.day_name().map(DIAS)
    df['tipo_movimiento'] = df['movimiento'].map({'A': 'Arribo', 'D': 'Partida'}).fillna(df['movimiento'])
    return df

conectividad = pd.concat([
    leer_conectividad('conectividad_aerea_cabotaje.csv', 'Cabotaje'),
    leer_conectividad('conectividad_aerea_internacional.csv', 'Internacional'),
], ignore_index=True)

media_ocupacion_por_clase = conectividad.groupby('Clasificacion')['ocupacion_original'].transform('mean')
conectividad['ocupacion_fue_imputada'] = conectividad['ocupacion_original'].isna()
conectividad['ocupacion'] = conectividad['ocupacion_original'].fillna(media_ocupacion_por_clase)

umbral_alta_ocupacion = conectividad.groupby('Clasificacion')['ocupacion'].median()
conectividad['umbral_alta_ocupacion'] = conectividad['Clasificacion'].map(umbral_alta_ocupacion)
conectividad['alta_ocupacion'] = conectividad['ocupacion'] >= conectividad['umbral_alta_ocupacion']

anac = pd.concat([
    leer_anac('202512-informe-ministerio-actualizado-dic-final.csv'),
    leer_anac('202604-informe-ministerio.csv'),
], ignore_index=True)
anac_2025 = anac[anac['anio'] == 2025].copy()

failbondi_raw = cargar_failbondi(DATA_DIR / 'failbondi-2026-05-20-dump.json')
failbondi = failbondi_raw[failbondi_raw['fecha_programada'] <= FECHA_CORTE_FAILBONDI].copy()
failbondi_limpio = failbondi[failbondi['delta_minutos'].between(-180, 360)].copy()

print('Datos cargados y preparados.')

## Diferencia entre Pasajeros y PAX en ANAC
No son lo mismo en este dataset.
Pasajeros: Cuenta pasajeros asociados a un Aterrizaje o a un Despegue. Es una variable de operación/movimiento. 
PAX: En vuelos internacionales coincide con Pasajeros. En vuelos domésticos suele ser aproximadamente la mitad de Pasajeros, porque el sistema puede registrar el viaje dentro del país en dos movimientos nacionales: salida y llegada. 
Ejemplo real del archivo ANAC: si un movimiento doméstico tiene Pasajeros = 167, aparece PAX = 83.5. En internacional, si Pasajeros = 281, aparece PAX = 281.
Por eso no se mezclan como sinónimos:
- Para **demanda agregada** se usa PAX / pasajeros_equivalentes.
- Para **pasajeros asociados a un movimiento** se usa Pasajeros / pasajeros_movimiento.
- Para **ocupación** se usa Pasajeros / Asientos, porque la ocupación corresponde al movimiento registrado.


In [ ]:
comparacion_pax = anac.groupby('Clasificación Vuelo').agg(
    pasajeros_movimiento=('pasajeros_movimiento', 'sum'),
    pasajeros_equivalentes=('pasajeros_equivalentes', 'sum'),
).reset_index()
comparacion_pax['relacion_pasajeros_sobre_pax'] = (
    comparacion_pax['pasajeros_movimiento'] / comparacion_pax['pasajeros_equivalentes']
)

muestra_pax = anac[[
    'Fecha UTC',
    'Tipo de Movimiento',
    'Clasificación Vuelo',
    'pasajeros_movimiento',
    'pasajeros_equivalentes',
]].head(8)

print('Comparación acumulada por clasificación')
display(comparacion_pax)

print('Ejemplos de registros reales')
display(muestra_pax)

print(
    'Conclusión: en ANAC, Pasajeros y PAX no son sinónimos. '
    'En internacional coinciden, pero en doméstico Pasajeros es casi el doble de PAX. '
    'Por eso el análisis de demanda agregada usa PAX como pasajeros_equivalentes, '
    'mientras que Pasajeros queda como pasajeros_movimiento.'
)


# Pregunta 1: ¿Cuál es el volumen y alcance temporal real de cada fuente?

In [ ]:
resumen_datasets = pd.DataFrame([
    {
        'fuente': 'Conectividad',
        'filas': len(conectividad),
        'columnas': conectividad.shape[1],
        'fecha_inicio': conectividad['fecha'].min().date(),
        'fecha_fin': conectividad['fecha'].max().date(),
        'uso_en_analisis': 'Serie 2019-2026; para tendencias anuales se excluye 2026 parcial',
    },
    {
        'fuente': 'ANAC aterrizajes/despegues',
        'filas': len(anac),
        'columnas': anac.shape[1],
        'fecha_inicio': anac['Fecha UTC'].min().date(),
        'fecha_fin': anac['Fecha UTC'].max().date(),
        'uso_en_analisis': 'Se usa 2025 completo para hora, día y mes',
    },
    {
        'fuente': 'Failbondi recortado al dump',
        'filas': len(failbondi),
        'columnas': failbondi.shape[1],
        'fecha_inicio': failbondi['fecha_programada'].min().date(),
        'fecha_fin': failbondi['fecha_programada'].max().date(),
        'uso_en_analisis': 'Se excluyen vuelos posteriores al 20/05/2026',
    },
    {
        'fuente': 'Failbondi limpio para demoras',
        'filas': len(failbondi_limpio),
        'columnas': failbondi_limpio.shape[1],
        'fecha_inicio': failbondi_limpio['fecha_programada'].min().date(),
        'fecha_fin': failbondi_limpio['fecha_programada'].max().date(),
        'uso_en_analisis': 'Se conservan demoras entre -180 y 360 minutos',
    },
])

display(resumen_datasets)

filas_futuras_failbondi = len(failbondi_raw) - len(failbondi)
porcentaje_conservado_demoras = len(failbondi_limpio) / len(failbondi) * 100

print(
    f"Conclusión: el análisis combina fuentes con distinto alcance temporal. "
    f"ANAC se analiza sobre 2025 completo y Failbondi se recorta al 20/05/2026; "
    f"se excluyeron {filas_futuras_failbondi:,} vuelos futuros del dump y se conservó "
    f"{porcentaje_conservado_demoras:.2f}% de los registros observados para demoras."
)

# Pregunta 2: ¿En qué horarios se concentra la mayor y menor demanda de pasajeros equivalentes?

Para evitar comparar períodos incompletos, se usa únicamente ANAC 2025 completo.

In [ ]:
pax_por_hora = (
    anac_2025.groupby('hora', dropna=True)['pasajeros_equivalentes']
    .sum()
    .reindex(range(24), fill_value=0)
    .reset_index()
)
pax_por_hora.columns = ['hora', 'pasajeros_equivalentes']

# Alias de compatibilidad: evita errores si quedó una ejecución vieja buscando PAX.
pax_por_hora['PAX'] = pax_por_hora['pasajeros_equivalentes']

hora_max = pax_por_hora.loc[pax_por_hora['pasajeros_equivalentes'].idxmax()]
hora_min = pax_por_hora.loc[pax_por_hora['pasajeros_equivalentes'].idxmin()]

display(pax_por_hora[['hora', 'pasajeros_equivalentes']])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(pax_por_hora['hora'], pax_por_hora['pasajeros_equivalentes'], marker='o', linewidth=2)
ax.scatter([hora_max['hora']], [hora_max['pasajeros_equivalentes']], color='green', s=80, label='Mayor demanda')
ax.scatter([hora_min['hora']], [hora_min['pasajeros_equivalentes']], color='red', s=80, label='Menor demanda')
ax.set_title('Pasajeros equivalentes por hora - ANAC 2025')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Pasajeros equivalentes')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"Conclusión: la mayor demanda se concentra a las {int(hora_max['hora'])}:00 "
    f"con {hora_max['pasajeros_equivalentes']:,.0f} pasajeros equivalentes; la menor ocurre a las {int(hora_min['hora'])}:00 "
    f"con {hora_min['pasajeros_equivalentes']:,.0f} pasajeros equivalentes. Esto sirve para identificar franjas donde reforzar capacidad operativa."
)


# Pregunta 3: ¿Qué días de la semana tienen más movimiento?

In [ ]:
pax_por_dia = (
    anac_2025.groupby('dia_semana')['pasajeros_equivalentes']
    .sum()
    .reindex(ORDEN_DIAS)
    .reset_index()
)

dia_max = pax_por_dia.loc[pax_por_dia['pasajeros_equivalentes'].idxmax()]
dia_min = pax_por_dia.loc[pax_por_dia['pasajeros_equivalentes'].idxmin()]

display(pax_por_dia)

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['#2f7fb8' if dia not in [dia_max['dia_semana'], dia_min['dia_semana']] else '#2ca02c' if dia == dia_max['dia_semana'] else '#d62728' for dia in pax_por_dia['dia_semana']]
ax.bar(pax_por_dia['dia_semana'], pax_por_dia['pasajeros_equivalentes'], color=colores)
ax.set_title('Pasajeros equivalentes por día - ANAC 2025')
ax.set_xlabel('Día de la semana')
ax.set_ylabel('Pasajeros equivalentes')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print(
    f"Conclusión: el día con más movimiento es {dia_max['dia_semana']} "
    f"({dia_max['pasajeros_equivalentes']:,.0f} pasajeros equivalentes) y el menor es {dia_min['dia_semana']} "
    f"({dia_min['pasajeros_equivalentes']:,.0f} pasajeros equivalentes). La demanda semanal es relativamente pareja, pero el ranking ayuda a planificar recursos."
)

# Pregunta 4: ¿Qué meses de 2025 presentan mayor y menor cantidad de pasajeros equivalentes?

In [ ]:
pax_por_mes = (
    anac_2025.groupby('mes')['pasajeros_equivalentes']
    .sum()
    .reindex(range(1, 13), fill_value=0)
    .reset_index()
)

mes_max = pax_por_mes.loc[pax_por_mes['pasajeros_equivalentes'].idxmax()]
mes_min = pax_por_mes.loc[pax_por_mes['pasajeros_equivalentes'].idxmin()]

nombres_meses = {v: k for k, v in MESES.items()}
pax_por_mes['mes_nombre'] = pax_por_mes['mes'].map(nombres_meses)

display(pax_por_mes[['mes', 'mes_nombre', 'pasajeros_equivalentes']])

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(pax_por_mes['mes_nombre'], pax_por_mes['pasajeros_equivalentes'], color='#4c78a8')
ax.set_title('Pasajeros equivalentes por mes - ANAC 2025')
ax.set_xlabel('Mes')
ax.set_ylabel('Pasajeros equivalentes')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print(
    f"Conclusión: el mes con mayor movimiento fue {nombres_meses[int(mes_max['mes'])]} "
    f"({mes_max['pasajeros_equivalentes']:,.0f} pasajeros equivalentes) y el menor fue {nombres_meses[int(mes_min['mes'])]} "
    f"({mes_min['pasajeros_equivalentes']:,.0f} pasajeros equivalentes). Esta estacionalidad es útil para anticipar capacidad y demanda turística."
)

# Pregunta 5: ¿Qué rutas concentran más pasajeros en cabotaje, internacional y total?

Se separa el ranking total, cabotaje e internacional para no mezclar mercados con comportamientos distintos.

In [ ]:
top_rutas = conectividad.groupby(['Clasificacion', 'Ruta']).agg(
    vuelos=('Vuelos', 'sum'),
    pasajeros=('Pasajeros', 'sum'),
    asientos=('Asientos', 'sum'),
).reset_index()
top_rutas['ocupacion'] = np.where(top_rutas['asientos'] > 0, top_rutas['pasajeros'] / top_rutas['asientos'], np.nan)

top_total = (
    top_rutas.groupby('Ruta').agg(
        vuelos=('vuelos', 'sum'),
        pasajeros=('pasajeros', 'sum'),
        asientos=('asientos', 'sum')
    )
    .reset_index()
)
top_total['ocupacion'] = top_total['pasajeros'] / top_total['asientos']
top_total = top_total.sort_values('pasajeros', ascending=False).head(10)

top_cabotaje = top_rutas[top_rutas['Clasificacion'] == 'Cabotaje'].sort_values('pasajeros', ascending=False).head(10)
top_internacional = top_rutas[top_rutas['Clasificacion'] == 'Internacional'].sort_values('pasajeros', ascending=False).head(10)

print('Top 10 total')
display(top_total)
print('Top 10 cabotaje')
display(top_cabotaje)
print('Top 10 internacional')
display(top_internacional)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, datos, titulo in [
    (axes[0], top_total.sort_values('pasajeros'), 'Top total'),
    (axes[1], top_cabotaje.sort_values('pasajeros'), 'Top cabotaje'),
    (axes[2], top_internacional.sort_values('pasajeros'), 'Top internacional'),
]:
    ax.barh(datos['Ruta'], datos['pasajeros'])
    ax.set_title(titulo)
    ax.set_xlabel('Pasajeros')
plt.tight_layout()
plt.show()

ruta_total = top_total.iloc[0]
ruta_cabotaje = top_cabotaje.iloc[0]
ruta_internacional = top_internacional.iloc[0]
print(
    f"Conclusión: la ruta con mayor volumen total es {ruta_total['Ruta']} "
    f"({ruta_total['pasajeros']:,.0f} pasajeros). En cabotaje lidera {ruta_cabotaje['Ruta']} "
    f"y en internacional {ruta_internacional['Ruta']}. Estas rutas son candidatas naturales para reforzar capacidad, frecuencias o seguimiento operativo."
)

# Pregunta 6: ¿Cómo se comporta la ocupación y cómo se define alta ocupación?

La alta ocupación se define con la mediana de cada clasificación. Esto evita usar un umbral único que podría no representar igual a cabotaje e internacional.

In [ ]:
resumen_ocupacion = conectividad.groupby('Clasificacion').agg(
    registros=('Ruta', 'count'),
    vuelos=('Vuelos', 'sum'),
    pasajeros=('Pasajeros', 'sum'),
    asientos=('Asientos', 'sum'),
    ocupacion_media=('ocupacion', 'mean'),
    ocupacion_mediana=('ocupacion', 'median'),
    registros_imputados=('ocupacion_fue_imputada', 'sum'),
    proporcion_alta_ocupacion=('alta_ocupacion', 'mean'),
).reset_index()

for columna in ['ocupacion_media', 'ocupacion_mediana', 'proporcion_alta_ocupacion']:
    resumen_ocupacion[columna] *= 100

umbral_df = (umbral_alta_ocupacion * 100).rename('umbral_alta_ocupacion_%').reset_index()

display(resumen_ocupacion)
display(umbral_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(resumen_ocupacion['Clasificacion'], resumen_ocupacion['ocupacion_media'], label='Media')
axes[0].scatter(resumen_ocupacion['Clasificacion'], resumen_ocupacion['ocupacion_mediana'], color='black', label='Mediana', zorder=3)
axes[0].set_title('Ocupación media y mediana')
axes[0].set_ylabel('Ocupación (%)')
axes[0].set_ylim(0, 100)
axes[0].legend()

axes[1].bar(resumen_ocupacion['Clasificacion'], resumen_ocupacion['proporcion_alta_ocupacion'], color='#59a14f')
axes[1].set_title('Registros con alta ocupación')
axes[1].set_ylabel('Proporción (%)')
axes[1].set_ylim(0, 100)
plt.tight_layout()
plt.show()

imputados = int(conectividad['ocupacion_fue_imputada'].sum())
print(
    f"Conclusión: se imputaron {imputados} valores nulos de ocupación con la media de su clasificación. "
    f"La ocupación es alta en ambos segmentos y el KPI de alta ocupación queda definido por la mediana propia de cada grupo, "
    f"no por un valor arbitrario."
)

# Pregunta 7: ¿Qué valores atípicos aparecen en la serie histórica y cómo se interpretan?

Se observan pasajeros anuales hasta 2025 para evitar que 2026 parcial distorsione la tendencia. Además se revisan días extremos para detectar caídas, picos y posibles registros atípicos.

In [ ]:
conectividad_anual = conectividad[conectividad['anio'].between(2019, 2025)].copy()
pasajeros_anuales = conectividad_anual.groupby(['anio', 'Clasificacion'])['Pasajeros'].sum().reset_index()

pasajeros_diarios = conectividad.groupby(['fecha', 'Clasificacion'])['Pasajeros'].sum().reset_index()
pasajeros_diarios_total = conectividad.groupby('fecha')['Pasajeros'].sum().reset_index()
pasajeros_diarios_total['anio'] = pasajeros_diarios_total['fecha'].dt.year

# Para detectar extremos comparables se excluye 2026 porque es un año parcial.
pasajeros_diarios_completo = pasajeros_diarios_total[pasajeros_diarios_total['anio'].between(2019, 2025)].copy()
q01 = pasajeros_diarios_completo['Pasajeros'].quantile(0.01)
q99 = pasajeros_diarios_completo['Pasajeros'].quantile(0.99)
atipicos_bajos = pasajeros_diarios_completo[pasajeros_diarios_completo['Pasajeros'] <= q01].sort_values('Pasajeros').head(10)
atipicos_altos = pasajeros_diarios_completo[pasajeros_diarios_completo['Pasajeros'] >= q99].sort_values('Pasajeros', ascending=False).head(10)

# Revisión puntual de 2024, mencionada en la discusión de clase.
pasajeros_2024_diarios = pasajeros_diarios_total[pasajeros_diarios_total['anio'] == 2024].copy()
extremos_2024_bajos = pasajeros_2024_diarios.sort_values('Pasajeros').head(5)
extremos_2024_altos = pasajeros_2024_diarios.sort_values('Pasajeros', ascending=False).head(5)

print('Pasajeros anuales por clasificación')
display(pasajeros_anuales)
print('Días con pasajeros muy bajos, excluyendo 2026 parcial')
display(atipicos_bajos[['fecha', 'Pasajeros']])
print('Días con pasajeros muy altos, excluyendo 2026 parcial')
display(atipicos_altos[['fecha', 'Pasajeros']])
print('Extremos bajos de 2024')
display(extremos_2024_bajos[['fecha', 'Pasajeros']])
print('Extremos altos de 2024')
display(extremos_2024_altos[['fecha', 'Pasajeros']])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for clasificacion, datos in pasajeros_anuales.groupby('Clasificacion'):
    axes[0].plot(datos['anio'], datos['Pasajeros'], marker='o', linewidth=2, label=clasificacion)
axes[0].set_title('Pasajeros anuales por clasificación (2019-2025)')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Pasajeros')
axes[0].legend()

axes[1].plot(pasajeros_diarios_completo['fecha'], pasajeros_diarios_completo['Pasajeros'], linewidth=0.8)
axes[1].axhline(q01, color='red', linestyle='--', label='Percentil 1')
axes[1].axhline(q99, color='green', linestyle='--', label='Percentil 99')
axes[1].set_title('Pasajeros diarios: extremos comparables (2019-2025)')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Pasajeros')
axes[1].legend()
plt.tight_layout()
plt.show()

pasajeros_2019 = pasajeros_anuales.loc[pasajeros_anuales['anio'] == 2019, 'Pasajeros'].sum()
pasajeros_2020 = pasajeros_anuales.loc[pasajeros_anuales['anio'] == 2020, 'Pasajeros'].sum()
caida_2020 = (1 - pasajeros_2020 / pasajeros_2019) * 100
max_2024 = extremos_2024_altos.iloc[0]
min_2024 = extremos_2024_bajos.iloc[0]

print(
    f"Conclusión: la caída de 2020 frente a 2019 fue de {caida_2020:.2f}% y se interpreta por el impacto de la pandemia. "
    f"En 2024, el menor día registrado fue {min_2024['fecha'].date()} con {min_2024['Pasajeros']:,.0f} pasajeros "
    f"y el mayor fue {max_2024['fecha'].date()} con {max_2024['Pasajeros']:,.0f}. "
    f"Esos extremos deben revisarse antes de usarlos para decisiones: pueden representar picos reales, estacionalidad o carga parcial."
)

# Pregunta 8: ¿Cuál es el comportamiento general de las demoras?

In [ ]:
resumen_demoras = failbondi_limpio['delta_minutos'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
media_demora = failbondi_limpio['delta_minutos'].mean()
mediana_demora = failbondi_limpio['delta_minutos'].median()
porcentaje_demora_15 = failbondi_limpio['demorado_15m'].mean() * 100
porcentaje_adelanto_5 = failbondi_limpio['adelantado_5m'].mean() * 100
hora_mas_vuelos_failbondi = failbondi_limpio['hora'].mode().iloc[0]

display(resumen_demoras)

rango_visual = (-60, 180)
demoras = failbondi_limpio['delta_minutos'].dropna()
demoras_grafico = demoras[demoras.between(*rango_visual)]
pesos = np.ones(len(demoras_grafico)) / len(demoras) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(
    demoras_grafico,
    bins=np.arange(rango_visual[0], rango_visual[1] + 5, 5),
    weights=pesos,
    color='#4c78a8',
    edgecolor='white',
)
ax.axvline(15, color='red', linestyle='--', linewidth=2, label='Umbral demora: 15 min')
ax.axvline(mediana_demora, color='black', linewidth=1.5, label=f'Mediana: {mediana_demora:.1f} min')
ax.axvline(media_demora, color='orange', linestyle=':', linewidth=2, label=f'Promedio: {media_demora:.1f} min')
ax.set_title('Distribución de demoras en minutos')
ax.set_xlabel('Demora en minutos')
ax.set_ylabel('Porcentaje de vuelos (%)')
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"Conclusión: la demora promedio es {media_demora:.2f} minutos, la mediana es {mediana_demora:.2f} minutos "
    f"y el {porcentaje_demora_15:.2f}% de los vuelos supera los 15 minutos de demora. La media mayor que la mediana indica una cola de demoras altas."
)

# Pregunta 9: ¿Las demoras cambian según tipo de movimiento y aerolínea?

In [ ]:
prob_demora_por_movimiento = failbondi_limpio.groupby('tipo_movimiento').agg(
    vuelos=('flight_id', 'count'),
    prob_demora_15m=('demorado_15m', 'mean'),
    demora_promedio_min=('delta_minutos', 'mean'),
    demora_mediana_min=('delta_minutos', 'median'),
).reset_index()
prob_demora_por_movimiento['prob_demora_15m'] *= 100

min_vuelos_aerolinea = 100
prob_demora_por_aerolinea = failbondi_limpio.groupby('aerolinea').agg(
    vuelos=('flight_id', 'count'),
    prob_demora_15m=('demorado_15m', 'mean'),
    demora_promedio_min=('delta_minutos', 'mean'),
    demora_mediana_min=('delta_minutos', 'median'),
).query('vuelos >= @min_vuelos_aerolinea').reset_index()
prob_demora_por_aerolinea['prob_demora_15m'] *= 100
top_demora_aerolinea = prob_demora_por_aerolinea.sort_values('prob_demora_15m', ascending=False).head(10)

print('Demora por tipo de movimiento')
display(prob_demora_por_movimiento)
print('Top aerolíneas por probabilidad de demora, con al menos 100 vuelos')
display(top_demora_aerolinea)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(prob_demora_por_movimiento['tipo_movimiento'], prob_demora_por_movimiento['prob_demora_15m'], color='#f58518')
axes[0].set_title('Demora > 15 min por tipo de movimiento')
axes[0].set_ylabel('Probabilidad (%)')
axes[0].set_ylim(0, 100)

datos_aero = top_demora_aerolinea.sort_values('prob_demora_15m')
axes[1].barh(datos_aero['aerolinea'], datos_aero['prob_demora_15m'], color='#e45756')
axes[1].set_title('Top 10 aerolíneas con mayor demora > 15 min')
axes[1].set_xlabel('Probabilidad (%)')
plt.tight_layout()
plt.show()

partidas = prob_demora_por_movimiento.loc[prob_demora_por_movimiento['tipo_movimiento'] == 'Partida', 'prob_demora_15m'].iloc[0]
arribos = prob_demora_por_movimiento.loc[prob_demora_por_movimiento['tipo_movimiento'] == 'Arribo', 'prob_demora_15m'].iloc[0]
aerolinea_top = top_demora_aerolinea.iloc[0]
print(
    f"Conclusión: las partidas tienen mayor probabilidad de demora ({partidas:.2f}%) que los arribos ({arribos:.2f}%). "
    f"Entre aerolíneas con al menos {min_vuelos_aerolinea} vuelos, la mayor probabilidad observada corresponde a {aerolinea_top['aerolinea']} "
    f"({aerolinea_top['prob_demora_15m']:.2f}%)."
)

# Pregunta 10: ¿Las demoras varían según hora del día y día de semana?

In [ ]:
demora_por_hora = failbondi_limpio.groupby('hora').agg(
    vuelos=('flight_id', 'count'),
    prob_demora_15m=('demorado_15m', 'mean'),
    demora_promedio_min=('delta_minutos', 'mean'),
).reset_index()
demora_por_hora['prob_demora_15m'] *= 100

demora_por_dia = failbondi_limpio.groupby('dia_semana').agg(
    vuelos=('flight_id', 'count'),
    prob_demora_15m=('demorado_15m', 'mean'),
    demora_promedio_min=('delta_minutos', 'mean'),
).reindex(ORDEN_DIAS).reset_index()
demora_por_dia['prob_demora_15m'] *= 100

print('Demoras por hora')
display(demora_por_hora)
print('Demoras por día')
display(demora_por_dia)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(demora_por_hora['hora'], demora_por_hora['prob_demora_15m'], marker='o')
axes[0].set_title('Probabilidad de demora > 15 min por hora')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Probabilidad (%)')
axes[0].set_xticks(range(24))

axes[1].bar(demora_por_dia['dia_semana'], demora_por_dia['prob_demora_15m'], color='#72b7b2')
axes[1].set_title('Probabilidad de demora > 15 min por día')
axes[1].set_xlabel('Día')
axes[1].set_ylabel('Probabilidad (%)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

hora_peor = demora_por_hora.loc[demora_por_hora['prob_demora_15m'].idxmax()]
dia_peor = demora_por_dia.loc[demora_por_dia['prob_demora_15m'].idxmax()]
print(
    f"Conclusión: la mayor probabilidad de demora por hora aparece a las {int(hora_peor['hora'])}:00 "
    f"({hora_peor['prob_demora_15m']:.2f}%). Por día de semana, el peor valor se observa el {dia_peor['dia_semana']} "
    f"({dia_peor['prob_demora_15m']:.2f}%)."
)

# Pregunta 11: ¿Qué probabilidades pueden estimarse con modelos binomial y Poisson?

Se usan dos enfoques probabilísticos, pero con una aclaración importante:

- **Binomial:** si se observan 20 vuelos, ¿cuál es la probabilidad de que al menos 10 tengan demora mayor a 15 minutos?
- **Poisson:** se contrasta si sirve para modelar la cantidad diaria de vuelos demorados. Para una Poisson, la media y la varianza deberían ser parecidas; si la varianza real es mucho mayor, el modelo no ajusta bien y conviene usar la frecuencia empírica.


In [ ]:
p_demora = failbondi_limpio['demorado_15m'].mean()
n = 20
k_min = 10
prob_binomial_al_menos_10 = sum(
    comb(n, k) * (p_demora ** k) * ((1 - p_demora) ** (n - k))
    for k in range(k_min, n + 1)
)

demoras_por_fecha = failbondi_limpio.groupby('fecha')['demorado_15m'].sum()
lambda_diaria = demoras_por_fecha.mean()
varianza_diaria = demoras_por_fecha.var()
indice_dispersion = varianza_diaria / lambda_diaria
umbral_poisson = 300
prob_poisson_mas_300 = poisson.sf(umbral_poisson, lambda_diaria)
prob_empirica_mas_300 = (demoras_por_fecha > umbral_poisson).mean()

probabilidades_modelos = pd.DataFrame([
    {
        'modelo': 'Binomial',
        'pregunta': 'P(al menos 10 demoras en 20 vuelos)',
        'parametros': f'n={n}, p={p_demora:.4f}',
        'probabilidad': prob_binomial_al_menos_10,
        'porcentaje': prob_binomial_al_menos_10 * 100,
        'interpretacion': 'Modelo útil como aproximación simple',
    },
    {
        'modelo': 'Poisson teórica',
        'pregunta': 'P(más de 300 demoras en un día)',
        'parametros': f'lambda={lambda_diaria:.2f}',
        'probabilidad': prob_poisson_mas_300,
        'porcentaje': prob_poisson_mas_300 * 100,
        'interpretacion': 'No ajusta bien: hay sobredispersión',
    },
    {
        'modelo': 'Frecuencia empírica',
        'pregunta': 'P(más de 300 demoras en un día)',
        'parametros': 'datos observados',
        'probabilidad': prob_empirica_mas_300,
        'porcentaje': prob_empirica_mas_300 * 100,
        'interpretacion': 'Más representativa para estos datos',
    },
])

display(probabilidades_modelos)
display(demoras_por_fecha.describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(demoras_por_fecha, bins=30, color='#b279a2', edgecolor='white')
ax.axvline(lambda_diaria, color='black', linewidth=2, label=f'Media diaria: {lambda_diaria:.1f}')
ax.axvline(umbral_poisson, color='red', linestyle='--', linewidth=2, label='Umbral: 300')
ax.set_title('Distribución diaria de vuelos demorados')
ax.set_xlabel('Vuelos demorados por día')
ax.set_ylabel('Cantidad de días')
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"Conclusión: con p={p_demora:.4f}, la probabilidad binomial de que al menos 10 de 20 vuelos estén demorados es "
    f"{prob_binomial_al_menos_10 * 100:.2f}%. Para Poisson, la media diaria es {lambda_diaria:.2f} y la varianza "
    f"es {varianza_diaria:.2f}, con índice de dispersión {indice_dispersion:.2f}. Como la varianza es mucho mayor que la media, "
    f"Poisson no ajusta bien: estima {prob_poisson_mas_300 * 100:.2f}% para más de 300 demoras, pero la frecuencia empírica es "
    f"{prob_empirica_mas_300 * 100:.2f}%."
)

# Conclusiones finales

## Cierre del análisis

El problema trabajado fue identificar patrones de demanda, ocupación y puntualidad del transporte aéreo argentino para convertir datos operativos en conclusiones útiles. Para eso se integraron tres fuentes locales: conectividad aérea, movimientos aeroportuarios de ANAC y demoras de Failbondi.

Los datos muestran que la demanda no se distribuye de forma uniforme. En ANAC 2025, usando `PAX` como pasajeros equivalentes, la mayor concentración horaria se observa a las **16:00** y la menor a las **5:00**. A nivel semanal, el día de mayor movimiento es **lunes** y el menor es **martes**. A nivel mensual, **diciembre** concentra el mayor volumen y **junio** el menor. Estos resultados permiten anticipar necesidades de capacidad, personal y planificación operativa.

En conectividad, las rutas de mayor tráfico están muy concentradas: **Ciudad de Buenos Aires - San Carlos de Bariloche** lidera el ranking total. Separar cabotaje e internacional evita conclusiones mezcladas y permite ver qué rutas deberían recibir seguimiento prioritario según el mercado analizado. La ocupación es elevada en ambos segmentos: media de **81,43%** en cabotaje y **80,80%** en internacional; las medianas son **85,34%** y **85,39%**, respectivamente. El KPI de alta ocupación se definió con la mediana propia de cada clasificación, no con un umbral arbitrario.

La serie histórica evidencia una caída marcada en 2020, consistente con el impacto de la pandemia. Ese punto no debería eliminarse automáticamente porque tiene explicación de dominio y aporta contexto. En 2024 también aparecen días extremos: el menor registro fue el **09/05/2024** con **9.265 pasajeros** y el mayor fue el **21/12/2024** con **102.988 pasajeros**. Estos valores deben revisarse antes de usarlos para decisiones, porque pueden representar picos reales, estacionalidad o carga parcial.

El análisis de demoras es uno de los resultados más accionables. Luego del recorte y limpieza de Failbondi, la demora promedio es **19,24 minutos**, la mediana es **10 minutos** y el **38,49%** de los vuelos supera los 15 minutos de demora. Las partidas presentan una probabilidad de demora mayor que los arribos (**47,45%** contra **19,93%**), lo que sugiere que las acciones de mejora deberían concentrarse especialmente en procesos previos a la salida.

Desde el punto de vista probabilístico, si se observan 20 vuelos, la probabilidad de que al menos 10 estén demorados es **20,24%**. Además, la cantidad diaria de demoras muestra sobredispersión: la media es **251,61** y la varianza es mucho mayor. Por eso, Poisson no ajusta bien para este caso: estima **0,14%** para superar 300 demoras en un día, mientras que la frecuencia empírica observada es **27,77%**. Este contraste es útil porque muestra cuándo un modelo teórico no debe usarse para decidir sin validar sus supuestos.

En síntesis, el sistema aéreo analizado muestra alta ocupación, demanda concentrada en determinados horarios y rutas, y un problema de demoras más fuerte en partidas que en arribos. Las decisiones más razonables a partir de este análisis serían reforzar capacidad en rutas líderes, planificar recursos según hora, día y mes de demanda, monitorear rutas y aerolíneas con alta demora, y profundizar el estudio de valores atípicos antes de convertirlos en reglas operativas.


# Limitaciones del análisis

- Las fuentes tienen distinto alcance temporal y distinto nivel de agregación.
- ANAC solo cuenta con 2025 completo y parte de 2026 en los archivos locales disponibles; por eso las comparaciones por hora, día y mes usan 2025.
- Failbondi no necesariamente representa todos los vuelos del país y puede tener sesgos propios de la forma en que recopila información.
- El recorte de demoras entre `-180` y `360` minutos mejora la estabilidad del análisis, pero deja fuera casos extremos que podrían investigarse por separado.
- La imputación de ocupación afecta pocos registros, pero se documenta porque introduce valores estimados.
- La binomial se usa como aproximación simple. La Poisson se contrasta explícitamente y no se toma como modelo válido cuando hay sobredispersión.
- No se incorporaron variables externas como clima, feriados, eventos turísticos o conflictos operativos, que podrían explicar parte de las demoras y picos de demanda.

# Bibliografía

- Administración Nacional de Aviación Civil (ANAC). Datos de aterrizajes y despegues publicados en datos.gob.ar.
- Datos Argentina. Dataset de transporte aéreo y movimientos aeroportuarios: https://www.datos.gob.ar/
- Tableros Yvera. Dataset de conectividad aérea: https://tableros.yvera.tur.ar/conectividad/
- Failbondi. Información de vuelos y demoras: https://failbondi.fail/acerca
- Material de clase de Matemática IV e Introducción a Ciencia de Datos.